Environment Specifications:

Disk size: 2,500 GB

CPUs: 4, RAM: 26

GPU: NVIDIa Tesla T4: 1

In [ ]:
import os 
BUCKET = os.environ['WORKSPACE_BUCKET']
print(BUCKET)

In [ ]:
%%bash
cd /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8
GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/broadinstitute/ml4h.git
cd ml4h
GIT_LFS_SKIP_SMUDGE=1 git switch -f sf_trans
GIT_LFS_SKIP_SMUDGE=1 git pull
pip install .

In [ ]:
! pip install nvidia-cudnn-cu11==9.8.0.87

In [ ]:
import os 
BUCKET = os.environ['WORKSPACE_BUCKET']
print(BUCKET)

import sys
sys.path.append('/home/jupyter/')

import os
import sys
import pandas as pd
import numpy as np
import random
import glob
import h5py
import matplotlib.pyplot as plt
import json

%matplotlib inline

# Package Installation

# Copying Files

In [ ]:
! gsutil -m cp -r $BUCKET/multimodal_ml_baselines/transformer/input/prevalent /home/jupyter/workspaces/multimodalaifordiseaseriskpredictioninallofusparticipantsv8/

In [ ]:
from ml4h.recipes import train_transformer_on_parquet, train_transformer_on_parquet_fast
from ml4h.arguments import parse_args

In [ ]:
#train_transformer_on_parquet_fast(args)
phecodes = [ 'CV_417_2','CV_417_3', 
            'CV_404_2', 'CV_416_2',
           'CV_401',
           'MB_286', 
    'MB_288', 'EM_236_1', 'EM_202_2', 'NS_333_1']

In [ ]:
df.head()

In [ ]:
df = pd.read_csv('all_phe_results_concat.csv')

df.head()
for phecode in df.phecode.unique().tolist():

    m = df[df.phecode == phecode].auroc.max()
    print(f'{phecode} : {m}')

In [ ]:
from pathlib import Path
import json

base_path = Path("./runs/")

json_data = []

for file_path in base_path.rglob("metrics_*.json"):
    if 'prevalent' not in str(file_path):
        continue
    with open(file_path, "r") as f:
        json_data.extend(json.load(f))

print(f"Loaded {len(json_data)} files.")

In [ ]:
df2 = pd.DataFrame(json_data)
df2[df2.Task=='CV_446_2'].head(16)

In [ ]:
#df = pd.read_csv('all_phe_results_concat.csv')
#phecodes = [ 'CV_417_3', 'CV_404_2', 'CV_401','MB_286', 'MB_288', 'EM_236_1',  'NS_333_1']
diff=0
for phecode in sorted(df2.Task.unique()):
#     if 'BI' in phecode or 'CV_446_2' in phecode or 'SS' in phecode:
#         continue
    m = df[df.phecode == phecode].auroc.max()
    mt = df2[(df2.Metric=="auROC")&(df2.Task==phecode)].Score.max()
    diff += mt-m
    print(f'{phecode} Baseline : {m:0.3f} Transformer: {mt:0.3f}, cum diff: {diff:0.3f}')

## Copying QC participants without EHR data

In [ ]:
#df = pd.read_csv('all_phe_results_concat.csv')
#phecodes = [ 'CV_417_3', 'CV_404_2', 'CV_401','MB_286', 'MB_288', 'EM_236_1',  'NS_333_1']
diff=0
for phecode in sorted(df2.Task.unique()):
#     if 'BI' in phecode or 'CV_446_2' in phecode or 'SS' in phecode:
#         continue
    m = df[(df.phecode == phecode)&(df.method.str.strip()=='HR, Steps, Sleep, Age&Sex&Bmi')].auroc.max()
    mt = df2[(df2.Metric=="auROC")&(df2.Task==phecode)].Score.max()
    diff += mt-m
    print(f'{phecode} Baseline : {m:0.3f} Transformer: {mt:0.3f}, cum diff: {diff:0.3f}')

In [ ]:
df.head(25)

In [ ]:
print(plot_df)

In [ ]:
%matplotlib inline

In [ ]:
plt.close('all')

In [ ]:
results = []

for phecode in sorted(df2.Task.unique()):
#     if 'BI' in phecode or 'CV_446_2' in phecode or 'SS' in phecode:
#         continue
    baseline = df[(df.phecode == phecode)&(df.method.str.strip()=='HR, Steps, Sleep, Age&Sex&Bmi')].auroc.max()
    transformer = df2[
        (df2.Metric == "auROC") &
        (df2.Task == phecode)
    ].Score.max()

    results.append({
        "phecode": phecode,
        "Baseline": baseline,
        "Transformer": transformer
    })

plot_df = pd.DataFrame(results)

In [ ]:
x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(x - width/2, plot_df["Baseline"], width, label="Baseline HR, Steps, Sleep, Age&Sex&Bmi Model")
plt.bar(x + width/2, plot_df["Transformer"], width, label="Transformer 3 Year Window")

plt.xticks(x, plot_df["phecode"], rotation=45, ha="right")
plt.ylabel("Performance")
plt.title("Baseline vs Transformer Performance by Phecode")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plot_df.head(50)